
# N07: Decision Rule Prior-Correction (Post-Processing)
**Objective:** Breach the 0.95114 boundary strictly by mathematically correcting the output probabilities using Prior-Correction scaling factors optimized via Nelder-Mead to maximize Balanced Accuracy.


In [ ]:

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
import scipy.optimize as opt

warnings.filterwarnings('ignore')

ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
SEED = 2026
N_FOLDS = 5


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Feature Engineering
for df in [train_df, test_df]:
    df['sleep_bin'] = pd.qcut(df['sleep_duration'], q=5, duplicates='drop').astype(str)
    df['stress_sleep_interact'] = df['stress_level'].astype(str) + '_' + df['sleep_bin']

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[TARGET_COL])

X_train_raw = train_df[cat_cols + num_cols].copy()
X_test_raw = test_df[cat_cols + num_cols].copy()

for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('Missing').astype(str)

num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(train_df[num_cols])
X_test_num_raw = num_imputer.transform(test_df[num_cols])


In [ ]:

# 2. Probability Matrix Generation (Top 3 Target-Encoded Architectures)
oof_probs = np.zeros((len(train_df), 3))
tst_probs = np.zeros((len(test_df), 3))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print("--- Phase 1: Generating OOF and Test Probability Matrices ---")

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[tr_i], X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = X_train_num_raw[tr_i], X_train_num_raw[va_i]
    y_tr, y_va = y_train[tr_i], y_train[va_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    
    class_counts = np.bincount(y_tr)
    weights = [len(y_tr) / (len(class_counts) * c) for c in class_counts]
    
    # 1. CatBoost
    model_cb = CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=6, class_weights=weights, random_seed=SEED+fold, verbose=0, task_type="CPU")
    model_cb.fit(X_tr_full, y_tr)
    fold_oof_cb = model_cb.predict_proba(X_va_full)
    fold_tst_cb = model_cb.predict_proba(X_te_full)
    
    # 2. XGBoost
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight('balanced', y_tr)
    model_xgb = XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=6, random_state=SEED+fold, n_jobs=-1, objective='multi:softprob', eval_metric='mlogloss', early_stopping_rounds=50)
    model_xgb.fit(X_tr_full, y_tr, sample_weight=sample_weights, eval_set=[(X_va_full, y_va)], verbose=0)
    fold_oof_xgb = model_xgb.predict_proba(X_va_full)
    fold_tst_xgb = model_xgb.predict_proba(X_te_full)
    
    # 3. HistGradientBoosting (Using sample_weight in fit)
    model_hgb = HistGradientBoostingClassifier(max_iter=1000, learning_rate=0.03, max_depth=6, random_state=SEED+fold)
    model_hgb.fit(X_tr_full, y_tr, sample_weight=sample_weights)
    fold_oof_hgb = model_hgb.predict_proba(X_va_full)
    fold_tst_hgb = model_hgb.predict_proba(X_te_full)
    
    # Uniform Blend for this fold
    fold_oof_blend = (fold_oof_cb + fold_oof_xgb + fold_oof_hgb) / 3.0
    fold_tst_blend = (fold_tst_cb + fold_tst_xgb + fold_tst_hgb) / 3.0
    
    oof_probs[va_i] = fold_oof_blend
    tst_probs += fold_tst_blend / N_FOLDS
    
    print(f"Fold {fold + 1}/{N_FOLDS} completed. Uncalibrated Balanced Acc: {balanced_accuracy_score(y_va, fold_oof_blend.argmax(axis=1)):.5f}")

uncalibrated_score = balanced_accuracy_score(y_train, oof_probs.argmax(axis=1))
print(f"\nBase Uncalibrated Ensemble OOF Balanced Accuracy: {uncalibrated_score:.5f}")


In [ ]:

# 3. Phase 2: Decision Rule Prior-Correction (Optimizer)
print("\n--- Phase 2: Nelder-Mead Threshold Calibration ---")

def objective_function(weights, oof_matrix, y_true):
    # Scale probabilities by weights
    scaled_probs = oof_matrix * weights
    preds = scaled_probs.argmax(axis=1)
    # We want to maximize balanced accuracy, so we return negative
    return -balanced_accuracy_score(y_true, preds)

# Initialize weights to 1.0
initial_weights = [1.0, 1.0, 1.0]

result = opt.minimize(
    objective_function, 
    x0=initial_weights, 
    args=(oof_probs, y_train), 
    method='Nelder-Mead',
    options={'maxiter': 1000, 'xatol': 1e-4}
)

optimal_weights = result.x
calibrated_oof_preds = (oof_probs * optimal_weights).argmax(axis=1)
calibrated_score = balanced_accuracy_score(y_train, calibrated_oof_preds)

print(f"Optimal Prior-Correction Weights: {optimal_weights}")
print(f"Calibrated OOF Balanced Accuracy: {calibrated_score:.5f}")
print(f"Net Gain from Post-Processing: {calibrated_score - uncalibrated_score:.5f}")


In [ ]:

# 4. Phase 3: Project onto Test Set
calibrated_tst_preds = (tst_probs * optimal_weights).argmax(axis=1)

sub_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].astype("int64"),
    TARGET_COL: [CLASSES[i] for i in calibrated_tst_preds]
})

sub_df.to_csv("submission.csv", index=False)
print("\nExported Prior-Corrected submission.csv")
